# Fabric — migration du runtime Spark 1.3 → 2.0

Ce notebook **Python** utilise le plan de contrôle Fabric : aucun calcul Spark et aucune modification des données. `INVENTAIRE` n'envoie que des GET et ne crée aucun fichier. Il couvre les espaces visibles par l'identité d'exécution, pas le tenant entier. Les espaces non Admin restent en lecture seule ; transmettez le résultat à leurs admins ou utilisez une identité d'automatisation approuvée et explicitement Admin dans chaque espace cible. Une capacité inactive est signalée sans tentative de réactivation.

1. Lancez `INVENTAIRE` et relevez les GUID. `CIBLES` vide signifie tous les espaces visibles **uniquement en inventaire**. `CANDIDAT` n'est pas une garantie de compatibilité.
2. Validez un pilote hors production puis les notebooks, jobs, environnements, pipelines et consommateurs Delta/Direct Lake. Pour les espaces avec Lakehouse ou autres dépendances, renseignez aussi `VALIDES`. Un environnement par défaut attaché reste hors automatisation : testez et publiez son runtime séparément, de même pour les environnements propres aux items.
3. Pour une vague approuvée : `MODE = 'MIGRATION'`, `CONFIRMATION = 'MIGRER 1.3 VERS 2.0'`, `CIBLES` non vide et `JOURNAL` vers un dossier Lakehouse OneLake existant et protégé (URI ABFSS absolue contenant `/Files/`). N'exécutez pas deux vagues simultanément : l'API n'expose pas de mise à jour conditionnelle documentée.
4. Pour un retour arrière : `MODE = 'RESTAURATION'`, `CONFIRMATION = 'RESTAURER 2.0 VERS 1.3'`, `CIBLES` et `RUN_A_RESTAURER` (GUID de la migration, dans le même `JOURNAL`). Seules les écritures confirmées sont restaurables, si les paramètres n'ont pas changé depuis. Un journal `avant` sans `apres` impose un contrôle manuel avant toute relance.

Une nouvelle session Spark est nécessaire pour valider la version effective. Le retour arrière ne restaure ni les artefacts ni le protocole Delta. Ne montez pas les protocoles pendant la validation ; vérifiez aussi les fonctionnalités activées par défaut sur les nouvelles tables, notamment les deletion vectors. Les rapports et modèles ne changent pas directement de runtime ; leurs données et actualisations peuvent être affectées en aval.

[Guide Microsoft de migration](https://learn.microsoft.com/fabric/data-engineering/runtime-2-0-migration-guide) · [API Spark Settings](https://learn.microsoft.com/rest/api/fabric/spark/workspace-settings/update-spark-settings)

L’inventaire tenant est optionnel (`INVENTAIRE_TENANT`). Il ne confère aucun droit Spark. La vague A contient les workspaces 1.3 candidats sans dépendance détectée ; la vague B nécessite une validation explicite.


In [ ]:
MODE = 'INVENTAIRE'  # INVENTAIRE, MIGRATION, RESTAURATION
CONFIRMATION = ''
TENANT_ATTENDU = ''  # GUID du tenant obligatoire pour toute écriture
CIBLES = []    # GUID des espaces de travail de cette vague
VALIDES = []   # GUID dont les dépendances ont été validées sur 2.0
EXCLUS = []    # GUID à ne jamais modifier ni analyser
JOURNAL = ''   # URI ABFSS en OneLake, dossier existant dans Files
RUN_A_RESTAURER = ''
INVENTAIRE_TENANT = False  # nécessite Administrateur Fabric et Tenant.Read.All (ou ReadWrite)


In [ ]:
import base64
import json
import time
from collections import Counter
from datetime import datetime, timezone
from urllib.parse import parse_qs, unquote, urlsplit
from uuid import UUID, uuid4

import requests

API = 'https://api.fabric.microsoft.com'
LECTURE_SEULE_BI = {'Report', 'SemanticModel', 'Dashboard', 'PaginatedReport'}
SOURCE, CIBLE = '1.3', '2.0'
PHRASES = {'MIGRATION': 'MIGRER 1.3 VERS 2.0', 'RESTAURATION': 'RESTAURER 2.0 VERS 1.3'}

def guids(valeurs):
    return {str(UUID(v)) for v in valeurs}

class ErreurAPI(RuntimeError):
    def __init__(self, reponse):
        self.statut = reponse.status_code
        try:
            details = reponse.json()
        except ValueError:
            details = {}
        if not isinstance(details, dict):
            details = {}
        self.code = details.get('errorCode', '-')
        super().__init__(f"HTTP {self.statut} | {self.code} | requestId={details.get('requestId', reponse.headers.get('requestId', '-'))}")

class Fabric:
    def __init__(self, jeton, tenant, session, pause=time.sleep):
        self.jeton = jeton
        self.tenant_attendu = str(UUID(tenant)) if tenant else None
        self.tenant = self.tenant_attendu
        self.session = session
        self.pause = pause
        self.token, self.exp = None, 0

    def en_tetes(self):
        if self.exp - time.time() < 300:
            token = self.jeton()
            segment = token.split('.')[1]
            claims = json.loads(base64.urlsafe_b64decode(segment + '=' * (-len(segment) % 4)))
            tenant_jeton = str(UUID(claims['tid']))
            if self.tenant is not None and tenant_jeton != self.tenant:
                raise RuntimeError('Le jeton Fabric ne correspond pas au tenant attendu.')
            self.token, self.exp = token, int(claims['exp'])
            self.tenant = tenant_jeton
        return {'Authorization': f'Bearer {self.token}', 'x-ms-fabric-skill': 'spark-cli'}

    def appel(self, methode, chemin, *, params=None, corps=None):
        if methode not in {'GET', 'PATCH'} or not chemin.startswith('/v1/'):
            raise ValueError('Appel Fabric non prévu.')
        # Un PATCH au résultat incertain ne doit jamais être rejoué automatiquement.
        essais = 4 if methode == 'GET' else 1
        for tentative in range(essais):
            try:
                r = self.session.request(methode, API + chemin, params=params, json=corps,
                                         headers=self.en_tetes(), timeout=(5, 25), allow_redirects=False)
            except (requests.Timeout, requests.ConnectionError):
                if tentative + 1 == essais:
                    raise
                self.pause(2 ** tentative)
                continue
            if r.status_code == 200:
                return r.json()
            if tentative + 1 == essais or r.status_code not in {429, 500, 502, 503, 504}:
                raise ErreurAPI(r)
            delai = r.headers.get('Retry-After', '')
            delai = int(delai) if delai.isascii() and delai.isdecimal() else 2 ** tentative
            if delai > 120:
                raise ErreurAPI(r)
            self.pause(delai)

    def lister(self, chemin, *, champ='value', **filtres):
        jeton, vus = None, set()
        while True:
            params = {**filtres, **({'continuationToken': jeton} if jeton else {})}
            page = self.appel('GET', chemin, params=params)
            if not isinstance(page.get(champ), list):
                raise ValueError('Réponse de pagination Fabric invalide.')
            yield from page[champ]
            if uri := page.get('continuationUri'):
                jeton = parse_qs(urlsplit(uri).query).get('continuationToken', [None])[0]
                if not jeton:
                    raise ValueError('Continuation Fabric sans jeton.')
            else:
                jeton = unquote(page.get('continuationToken') or '')
            if not jeton:
                return
            if jeton in vus:
                raise ValueError('Boucle de pagination Fabric.')
            vus.add(jeton)

def chemin_spark(ws):
    return f'/v1/workspaces/{ws}/spark/settings'

def environnement(config):
    env = config.get('environment')
    if not isinstance(env, dict) or not isinstance(env.get('runtimeVersion'), str) or not isinstance(env.get('name', ''), str):
        raise ValueError('Paramètres Spark incomplets : état non vérifiable.')
    return env['runtimeVersion'], env.get('name', '')

def normaliser_settings(config):
    _, nom = environnement(config)
    return dict(config, environment=dict(config['environment'], name=nom))

def admins(api):
    return {str(UUID(w['id'])) for w in api.lister('/v1/workspaces', roles='Admin')}


In [ ]:
def analyser(api, w, admin, valides, *, config=None):
    ws = str(UUID(w['id']))
    ligne = dict(workspace=w['displayName'], workspace_id=ws, admin=admin,
                 capacity_id=w.get('capacityId'), runtime=None, environnement=None, types_items=None)
    if w['type'] != 'Workspace' or not ligne['capacity_id']:
        return dict(ligne, decision='HORS_PERIMETRE')
    try:
        if config is None:
            config = api.appel('GET', chemin_spark(ws))
        runtime, nom_env = environnement(config)
        ligne.update(runtime=runtime, environnement=nom_env)
        if runtime == CIBLE:
            return dict(ligne, decision='DEJA_EN_2_0')
        if runtime != SOURCE:
            return dict(ligne, decision='HORS_RUNTIME_SOURCE')
        if nom_env:
            return dict(ligne, decision='ENVIRONNEMENT_A_TRAITER')
        if not admin:
            return dict(ligne, decision='DELEGUER_A_UN_ADMIN')
        types = Counter(i.get('type') or '(inconnu)' for i in api.lister(f'/v1/workspaces/{ws}/items', recursive='true'))
        ligne['types_items'] = dict(sorted(types.items()))
    except ErreurAPI as exc:
        decision = 'CAPACITE_INACTIVE' if exc.code == 'CapacityNotActive' else 'LECTURE_INCOMPLETE'
        if exc.statut == 403 and not admin and decision == 'LECTURE_INCOMPLETE':
            decision = 'ACCES_SPARK_REQUIS'
        return dict(ligne, decision=decision, detail=str(exc))
    except (requests.RequestException, ValueError) as exc:
        return dict(ligne, decision='LECTURE_INCOMPLETE', detail=str(exc))
    if types.keys() - LECTURE_SEULE_BI and ws not in valides:
        return dict(ligne, decision='VALIDATION_DEPENDANCES_REQUISE', vague='B')
    return dict(ligne, decision='CANDIDAT', vague='B' if types.keys() - LECTURE_SEULE_BI else 'A')

def inventaire(api, cibles, valides, exclus, *, tenant=False):
    avec_role = admins(api)
    visibles = {str(UUID(w['id'])): w for w in api.lister('/v1/workspaces')}
    if tenant:
        for w in api.lister('/v1/admin/workspaces', champ='workspaces', type='workspace', state='active'):
            if w.get('type') == 'Workspace' and w.get('state') == 'Active':
                visibles.setdefault(str(UUID(w['id'])), dict(w, displayName=w['name']))
    ids = sorted((cibles or visibles.keys()) - exclus)
    rapport = []
    for ws in ids:
        if ws not in visibles:
            rapport.append(dict(workspace_id=ws, decision='NON_VISIBLE'))
        else:
            rapport.append(analyser(api, visibles[ws], ws in avec_role, valides))
    return rapport, visibles


In [ ]:
def journaliser(fs, chemin, entree):
    if not fs.put(chemin, json.dumps(entree, sort_keys=True), False):
        raise RuntimeError(f'Écriture du journal impossible : {chemin}')

def preparer_restauration(fs, racine, run, ws, tenant, actuel):
    prefixe = f'{racine}/{run}/{ws}'
    avant = json.loads(fs.head(prefixe + '-avant.json'))
    apres = json.loads(fs.head(prefixe + '-apres.json'))
    identite = {'mode': 'MIGRATION', 'tenant': tenant, 'run': run, 'workspace_id': ws,
                'source': SOURCE, 'cible': CIBLE}
    if any(entry.get(k) != v for entry in (avant, apres) for k, v in identite.items()):
        raise RuntimeError(f'Journal incompatible : {ws}')
    if (apres.get('statut') != 'VERIFIE' or normaliser_settings(apres['settings']) != normaliser_settings(actuel)
            or environnement(avant['settings']) != (SOURCE, '') or environnement(actuel) != (CIBLE, '')):
        raise RuntimeError(f'Restauration bloquée : état courant différent ou migration non vérifiée ({ws}).')

def changer(api, fs, dossier, run, mode, ws, avant, source, cible):
    chemin = chemin_spark(ws)
    if environnement(avant) != (source, ''):
        raise RuntimeError(f'État source inattendu pour {ws}.')
    prefixe = f'{dossier}/{ws}'
    entree = dict(tenant=api.tenant, run=run, mode=mode, workspace_id=ws, source=source, cible=cible,
                  date_utc=datetime.now(timezone.utc).isoformat())
    journaliser(fs, prefixe + '-avant.json', dict(entree, settings=avant))
    try:
        api.appel('PATCH', chemin, corps={'environment': {'runtimeVersion': cible}})
        relu = api.appel('GET', chemin)
        attendu = dict(avant, environment=dict(avant['environment'], runtimeVersion=cible))
        if normaliser_settings(relu) != normaliser_settings(attendu):
            raise RuntimeError('La relecture ne correspond pas au seul changement attendu.')
        journaliser(fs, prefixe + '-apres.json', dict(entree, settings=relu, statut='VERIFIE'))
    except (ErreurAPI, requests.RequestException, RuntimeError, ValueError) as exc:
        raise RuntimeError(f'Vague arrêtée sur {ws}. Vérifiez les paramètres réels et {prefixe}-avant.json avant de relancer. {exc}') from exc

def racine_journal(fs, journal):
    racine = journal.rstrip('/')
    uri = urlsplit(racine)
    if (uri.scheme != 'abfss' or not uri.hostname or not uri.hostname.endswith('.dfs.fabric.microsoft.com')
            or '/Files/' not in uri.path + '/' or uri.query or uri.fragment):
        raise ValueError('JOURNAL doit désigner un dossier OneLake ABFSS sous Files.')
    if not fs.exists(racine):
        raise ValueError('Le dossier JOURNAL doit exister avant la vague.')
    return racine

def appliquer(api, fs, mode, confirmation, cibles, valides, exclus, journal, run_source):
    if mode not in PHRASES or PHRASES[mode] != confirmation or not cibles or (cibles & exclus):
        raise ValueError('Confirmez le mode et indiquez des cibles explicites, sans espaces exclus.')
    if not api.tenant_attendu:
        raise ValueError('TENANT_ATTENDU est obligatoire pour toute écriture.')
    racine = racine_journal(fs, journal)
    restauration = mode == 'RESTAURATION'
    if restauration:
        run_source = str(UUID(run_source))
    rapport, visibles = inventaire(api, cibles, valides, exclus)
    actives = {str(UUID(c['id'])) for c in api.lister('/v1/capacities') if c.get('state') == 'Active'}
    candidats = []
    for ligne in rapport:
        ws = ligne['workspace_id']
        decision = ligne['decision']
        if not restauration and decision == 'DEJA_EN_2_0' and ligne.get('admin'):
            print(f'{ws} : déjà en 2.0 ; aucun PATCH')
            continue
        if not ligne.get('admin') or decision != ('DEJA_EN_2_0' if restauration else 'CANDIDAT'):
            raise RuntimeError(f'Vague bloquée : {ws} — {decision} — {ligne.get("detail", "")}')
        if str(UUID(ligne['capacity_id'])) not in actives:
            raise RuntimeError(f'Capacité non confirmée active : {ws}.')
        candidats.append(ws)
    if not candidats:
        return
    run = str(uuid4())
    dossier = f'{racine}/{run}'
    if not fs.mkdirs(dossier):
        raise RuntimeError('Création du dossier de journal impossible.')
    print(f'Run : {run} | Journal : {dossier}')
    for ws in candidats:
        avant = api.appel('GET', chemin_spark(ws))
        if not restauration:
            decision = analyser(api, visibles[ws], True, valides, config=avant)['decision']
            if decision != 'CANDIDAT':
                raise RuntimeError(f'Vague interrompue : inventaire modifié pour {ws} ({decision}).')
        if restauration:
            preparer_restauration(fs, racine, run_source, ws, api.tenant, avant)
        source, cible = (CIBLE, SOURCE) if restauration else (SOURCE, CIBLE)
        changer(api, fs, dossier, run, mode, ws, avant, source, cible)
        print(f'{ws} : {source} → {cible} vérifié')


In [ ]:
import notebookutils

if MODE not in {'INVENTAIRE', *PHRASES}:
    raise ValueError('MODE inconnu.')

with requests.Session() as session:
    api = Fabric(lambda: notebookutils.credentials.getToken('pbi'), TENANT_ATTENDU, session)
    cibles, valides, exclus = map(guids, (CIBLES, VALIDES, EXCLUS))
    if MODE == 'INVENTAIRE':
        rapport, _ = inventaire(api, cibles, valides, exclus, tenant=INVENTAIRE_TENANT)
        print(f'SIMULATION — {len(rapport)} espace(s) analysé(s), aucun PATCH ni fichier créé.')
        print(dict(Counter(ligne['decision'] for ligne in rapport)))
        if rapport:
            import pandas as pd
            display(pd.DataFrame(rapport))
    else:
        appliquer(api, notebookutils.fs, MODE, CONFIRMATION, cibles, valides, exclus, JOURNAL, RUN_A_RESTAURER)
